# Hybrid PCA for GSF Nucleon Flux Uncertainties

GSF has 88 correlated spline parameters across four element groups. **Hybrid PCA** compresses them into 12 uncorrelated standard-normal components $\phi_k \sim \mathcal{N}(0, 1)$ while preserving marginal variances at every energy via a block-diagonal correction.

The decomposition lives in parameter space, so reduced Jacobians evaluate exactly at any energy grid — no interpolation. This notebook builds the decomposition step by step; for production use, `globalsplinefit.pca.HybridPCA` wraps the same construction.

**Method**: (1) flux-space eigendecomposition gives a rank-12 factor $L$ plus diagonal residual $D$ (and proton-neutron cross-term $D_{pn}$); (2) project back to parameter space via least-squares so $J_{\text{rel}} L_{\text{param}} = L$ exactly. The full flux covariance is then $L L^T + D$ (with $D_{pn}$ on the p-n cross-blocks).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from globalsplinefit import GSFEnergyPerNucleon

plt.rcParams["figure.dpi"] = 100

In [ ]:
# Energy grid and GSF model (Solar Cycle 24 average)
xkin = np.logspace(np.log10(1.0), 11, 300)
gsf = GSFEnergyPerNucleon(
    version="2025",
    default_time_interval=(200812, 202001),
    use_approximate_solar_cycle_average=True,
)

# Total proton and neutron fluxes and their uncertainties
y5_p = sum(gsf.p_and_n_flux(xkin, g)[0] for g in ("p", "He", "O", "Fe"))
y5_n = sum(gsf.p_and_n_flux(xkin, g)[1] for g in ("p", "He", "O", "Fe"))

y5cov_p = y5cov_n = 0.0
for g1 in ("p", "He", "O", "Fe"):
    for g2 in ("p", "He", "O", "Fe"):
        c = gsf.p_and_n_covariance(g1, g2, xkin)
        y5cov_p += c[0]
        y5cov_n += c[1]

sigma_p = np.sqrt(np.diag(y5cov_p))  # 1-sigma on total proton flux
sigma_n = np.sqrt(np.diag(y5cov_n))  # 1-sigma on total neutron flux

## Reference: full uncertainty

Total proton and neutron flux with the full 88-parameter $\pm 1\sigma$ band. The 12-component model must reproduce this.

In [ ]:
E = 2.7
fig, ax = plt.subplots(figsize=(9, 5))

for flux, sigma, col, label in [
    (y5_p, sigma_p, "C0", "Protons"),
    (y5_n, sigma_n, "C1", "Neutrons"),
]:
    ax.loglog(xkin, xkin**E * flux, color=col, lw=2, label=label)
    ax.fill_between(
        xkin,
        xkin**E * (flux - sigma),
        xkin**E * (flux + sigma),
        color=col,
        alpha=0.25,
    )

ax.set_xlabel("Kinetic energy per nucleon (GeV)")
ax.set_ylabel(r"$E^{2.7} \times \Phi$ [GeV$^{1.7}$ m$^{-2}$ s$^{-1}$ sr$^{-1}$]")
ax.set_title("GSF 2025 — total nucleon flux with $\\pm 1\\sigma$ band")
ax.legend()
plt.tight_layout()

## Build the 12-component model

The four groups (H, He, O\*, Fe\*) each contribute $[J_p; J_n]$ rows in a block-diagonal Jacobian. With 300 energies × 2 species × 4 groups the system is $(2400, 88)$.

Steps: (1) stack block-diagonal Jacobians; (2) assemble the $88 \times 88$ block covariance; (3) divide each row by the central flux to get the relative Jacobian; (4) form $\Sigma_{\text{flux}} = J_{\text{rel}} \Sigma_{\text{param}} J_{\text{rel}}^T$; (5) eigendecompose and add the diagonal corrections.

In [ ]:
groups = ["p", "He", "O*", "Fe*"]
par_vecs = []
jac_blocks = []
flux_parts = []
npar_trimmed = []

for g in groups:
    z = gsf.GROUP_NAMES[g]
    par_vecs.append(gsf.pars[z][1:-7])
    jp, jn = gsf.p_and_n_jacobian(xkin, g)
    jac_blocks.append(np.vstack([jp[:, 1:-3], jn[:, 1:-3]]))
    pn = gsf.p_and_n_flux(xkin, g)
    flux_parts.append(pn.ravel())  # (2*N,)
    npar_trimmed.append(jac_blocks[-1].shape[1])

par_stack = np.hstack(par_vecs)

# Block-diagonal Jacobian (2400, 88): each group's rows occupy their own column block
n_rows = sum(j.shape[0] for j in jac_blocks)
n_cols = sum(j.shape[1] for j in jac_blocks)
jac_stack = np.zeros((n_rows, n_cols))

row_off, col_off = 0, 0
for j in jac_blocks:
    r, c = j.shape
    jac_stack[row_off : row_off + r, col_off : col_off + c] = j
    row_off += r
    col_off += c

# Block covariance matrix (88, 88)
group_leaders = [gsf.GROUP_NAMES[g] for g in groups]
cov_stack = np.zeros((n_cols, n_cols))
col_i = 0
for i, zi in enumerate(group_leaders):
    ni = npar_trimmed[i]
    col_j = 0
    for j, zj in enumerate(group_leaders):
        nj = npar_trimmed[j]
        cov_key = (zi, zj)
        if cov_key in gsf.cov:
            cov_stack[col_i : col_i + ni, col_j : col_j + nj] = gsf.cov[cov_key][
                1:-3, 1:-3
            ]
        col_j += nj
    col_i += ni

central_flux = np.concatenate(flux_parts)

# Relative Jacobian (mask out zero-flux rows, e.g. neutron flux from H group)
nonzero = central_flux != 0.0
rel_jac = np.zeros_like(jac_stack)
rel_jac[nonzero] = jac_stack[nonzero] / central_flux[nonzero, np.newaxis]

# Relative flux covariance (2400 x 2400)
flux_rel_cov = rel_jac @ cov_stack @ rel_jac.T

n_energy = len(xkin)
n_groups = len(groups)
rpg = 2 * n_energy  # rows per group (proton + neutron)

print(f"Jacobian shape:      {jac_stack.shape}")
print(f"Covariance shape:    {cov_stack.shape}")
print(f"Central flux shape:  {central_flux.shape}")
print(f"Flux cov shape:      {flux_rel_cov.shape}")
print(f"Rows per group:      {rpg} ({n_energy} proton + {n_energy} neutron)")

In [ ]:
n_components = 12

# Step 1 — flux-space PCA: optimal rank-12 approximation
eigvals, eigvecs = np.linalg.eigh(flux_rel_cov)
idx = np.argsort(eigvals)[::-1]
eigvals, eigvecs = eigvals[idx], eigvecs[:, idx]

L = eigvecs[:, :n_components] * np.sqrt(np.maximum(eigvals[:n_components], 0.0))

# Same-block diagonal correction: exact variance minus low-rank contribution
D = np.maximum(np.diag(flux_rel_cov) - np.sum(L**2, axis=1), 0.0)

# Cross-term diagonal correction D_pn: captures the residual covariance
# between proton and neutron rows that the low-rank factor misses.
#
# For nuclei where Z ~ A-Z (He, O, Fe), the relative Jacobians satisfy
# J_rel_p ~ J_rel_n (correlation ~ 1.0), so the cross-term Cov(p,n) is as
# large as Cov(p,p). The low-rank factor captures only ~20% of this; the
# remaining ~80% lives in D_pn. Without it, total (p+n) errors are
# underestimated by ~30% for these groups.
D_pn = np.zeros(n_groups * n_energy)
for ig in range(n_groups):
    p_idx = np.arange(n_energy) + ig * rpg
    n_idx = p_idx + n_energy
    exact_pn_diag = flux_rel_cov[p_idx, n_idx]
    lr_pn_diag = np.sum(L[p_idx] * L[n_idx], axis=1)
    d_pn = exact_pn_diag - lr_pn_diag
    # Clamp for PSD safety: D_pn <= sqrt(D_p * D_n)
    d_pn = np.minimum(d_pn, np.sqrt(D[p_idx] * D[n_idx]))
    D_pn[ig * n_energy : (ig + 1) * n_energy] = d_pn

# Step 2 — project to parameter space via least-squares: J_rel @ L_param ≈ L
L_param, _, _, _ = np.linalg.lstsq(rel_jac, L, rcond=None)

# M is the reduced relative Jacobian at the training grid
M = rel_jac @ L_param  # (2400, 12) ≡ L (to machine precision)

print(f"Max reconstruction error |M - L|: {np.max(np.abs(M - L)):.2e}")
print(
    f"Diagonal preserved to: {np.max(np.abs(np.sum(M**2, axis=1) + D - np.diag(flux_rel_cov))):.2e}"
)
print(f"D_pn range: [{D_pn.min():.2e}, {D_pn.max():.2e}]")

## Variance captured

The flux covariance has rank ≤ 88; in practice the eigenvalue spectrum falls steeply. 12 components capture ~93% of $\mathrm{tr}(\Sigma_{\text{flux}})$; the remaining variance lives in $D$ (same-block) and $D_{pn}$ (cross-block). The hybrid combination reproduces marginal uncertainties and proton-neutron cross-covariances exactly.

In [ ]:
pos = eigvals > 0
cumvar = np.cumsum(eigvals[pos]) / eigvals[pos].sum()
r_vals = np.arange(1, pos.sum() + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.bar(
    r_vals[:30] - 0.5,
    eigvals[pos][:30] / eigvals[pos].sum() * 100,
    width=1,
    color="C0",
    alpha=0.7,
    label="Individual",
)
ax.step(
    r_vals[:30], cumvar[:30] * 100, where="post", color="C1", lw=2, label="Cumulative"
)
ax.axvline(n_components, color="k", ls="--", lw=1, label=f"r = {n_components}")
ax.axhline(cumvar[n_components - 1] * 100, color="k", ls=":", lw=1)
ax.set_xlabel("Component index")
ax.set_ylabel("Explained variance (%)")
ax.set_title("Variance per component")
ax.legend()
ax.set_xlim(0.5, 30.5)

ax = axes[1]
ax.plot(r_vals, cumvar * 100, "o-", ms=3, color="C0")
ax.axhline(99, color="C1", ls="--", lw=1, label="99 %")
ax.axvline(n_components, color="k", ls="--", lw=1, label=f"r = {n_components}")
ax.set_xlabel("Number of components r")
ax.set_ylabel("Cumulative explained variance (%)")
ax.set_title("Convergence")
ax.set_xlim(0, 40)
ax.legend()

plt.tight_layout()
print(f"Cumulative variance at r={n_components}: {cumvar[n_components - 1] * 100:.1f}%")
print(
    f"Variance in diagonal correction D:      {D.sum() / np.diag(flux_rel_cov).sum() * 100:.1f}%"
)

## The 12 components

Each component $\phi_k$ is an independent mode: setting $\phi_k = \pm 1$ gives a flux perturbation $\delta f / f = \pm M_k$. Component 1 is roughly an overall normalization; 2–3 are broadband spectral tilts; higher components describe progressively finer features.

Below: $f_{\text{central}} \times (1 \pm M_k)$ for the H-group proton row.

In [ ]:
# Extract proton and neutron central fluxes for the H group (group 0)
cf_p_H = central_flux[:n_energy]
cf_n_H = central_flux[n_energy:rpg]

fig, axes = plt.subplots(3, 4, figsize=(16, 9), sharex=True, sharey=True)
fig.subplots_adjust(hspace=0.08, wspace=0.08)

for k, ax in enumerate(axes.flat):
    # H group proton and neutron rows
    mk_p = M[:n_energy, k]
    mk_n = M[n_energy:rpg, k]

    # ±1σ in absolute flux
    ax.fill_between(
        xkin,
        xkin**E * cf_p_H * (1 + mk_p),
        xkin**E * cf_p_H * (1 - mk_p),
        color="C0",
        alpha=0.35,
        label="Protons" if k == 0 else None,
    )
    ax.fill_between(
        xkin,
        xkin**E * cf_n_H * (1 + mk_n),
        xkin**E * cf_n_H * (1 - mk_n),
        color="C1",
        alpha=0.35,
        label="Neutrons" if k == 0 else None,
    )

    # Central flux (thin line)
    ax.loglog(xkin, xkin**E * cf_p_H, color="C0", lw=0.8, alpha=0.6)
    ax.loglog(xkin, xkin**E * cf_n_H, color="C1", lw=0.8, alpha=0.6, ls="--")

    ax.set_xlim(1, 1e11)
    ax.text(
        0.95,
        0.05,
        f"c={k + 1}",
        transform=ax.transAxes,
        fontsize=9,
        ha="right",
        va="bottom",
    )
    if k == 0:
        ax.legend(fontsize=7, loc="upper right")

    if k % 4 != 0:
        ax.tick_params(labelleft=False)

for ax in axes[-1]:
    ax.set_xlabel("Energy (GeV)", fontsize=8)
for ax in axes[:, 0]:
    ax.set_ylabel(r"$E^{2.7} \times \Phi$", fontsize=8)

fig.suptitle(
    r"$\pm 1\sigma$ flux variation per component (H group, $\phi_k = \pm 1$)",
    fontsize=11,
)
plt.tight_layout()

## Flux ratio view

Same 12 components plotted as $1 \pm M_k(E)$, isolating the energy-dependent shape of each mode and showing proton vs. neutron differences from group composition.

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(8, 4), sharex=True)
fig.subplots_adjust(hspace=0.08, wspace=0.08)

for k, ax in enumerate(axes.flat):
    mk_p = M[:n_energy, k]
    mk_n = M[n_energy:rpg, k]

    ax.fill_between(
        xkin,
        1 + mk_p,
        1 - mk_p,
        color="C0",
        alpha=0.4,
        label="Protons" if k == 0 else None,
    )
    ax.fill_between(
        xkin,
        1 + mk_n,
        1 - mk_n,
        color="C1",
        alpha=0.4,
        label="Neutrons" if k == 0 else None,
    )
    ax.axhline(1, color="k", lw=0.6, ls=":")
    ax.set_xscale("log")
    ax.set_xlim(1, 1e11)
    if k != 0:
        ax.set_ylim(0.75, 1.25)
    ax.grid(True, which="both", ls=":", lw=0.5, alpha=0.7)
    ax.text(
        0.05,
        0.05,
        rf"$c={k + 1}$",
        transform=ax.transAxes,
        fontsize=9,
        ha="left",
        va="bottom",
    )
    if k == 0:
        ax.legend(fontsize=7)
    if k % 4 != 0:
        ax.tick_params(labelleft=False)

for ax in axes[-1]:
    ax.set_xlabel("Energy (GeV)", fontsize=8)
for ax in axes[:, 0]:
    ax.set_ylabel(r"$f/f_{\rm central}$", fontsize=8)

plt.tight_layout(h_pad=-0.1, w_pad=-0.5)

In [ ]:
from globalsplinefit.pca import HybridPCA

# Separate p/n model uncertainty for H group (group 0)
model_sigma_p_H = np.sqrt(np.sum(M[:n_energy] ** 2, axis=1) + D[:n_energy]) * cf_p_H
model_sigma_n_H = (
    np.sqrt(np.sum(M[n_energy:rpg] ** 2, axis=1) + D[n_energy:rpg]) * cf_n_H
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.loglog(xkin, sigma_p, color="C0", lw=2.5, label="Exact $\\sigma_p$ (total)")
ax.loglog(
    xkin,
    model_sigma_p_H,
    color="C0",
    lw=1.0,
    ls="--",
    label="Model $\\sigma_p$ (H group)",
)
ax.loglog(xkin, sigma_n, color="C1", lw=2.5, label="Exact $\\sigma_n$ (total)")
ax.loglog(
    xkin,
    model_sigma_n_H,
    color="C1",
    lw=1.0,
    ls="--",
    label="Model $\\sigma_n$ (H group)",
)
ax.set_xlabel("Energy (GeV)")
ax.set_ylabel(r"$1\sigma$ uncertainty (GeV$^{-1}$ m$^{-2}$ s$^{-1}$ sr$^{-1}$)")
ax.set_title("Uncertainty profiles (separate p/n)")
ax.legend(fontsize=8)

# Right panel: total (p+n) error ratio per group, using HybridPCA class
ax = axes[1]
pca = HybridPCA(gsf, n_components=12, energy_grid=xkin)
for g in groups:
    pca_err = pca.error(xkin, g)
    model_err = gsf.error(xkin, g)
    ax.semilogx(xkin, pca_err / model_err, lw=1.5, label=g)
ax.axhline(1.0, color="k", ls="--", lw=1)
ax.set_xlabel("Energy (GeV)")
ax.set_ylabel("PCA / Exact")
ax.set_title("Total (p+n) error ratio per group")
ax.set_ylim(0.99, 1.01)
ax.legend()

plt.tight_layout()

## Validation and Monte Carlo samples

Left: model $\sigma_p$ and $\sigma_n$ for the H group against the exact totals. Right: PCA-vs-exact total (p+n) error ratio per group — agrees to ~1%.

Then draw $\phi \sim \mathcal{N}(0, I_{12})$ and form Monte Carlo flux realizations.

In [ ]:
rng = np.random.default_rng(0)
n_samples = 2000

phi = rng.standard_normal((n_samples, n_components))
rel_var = phi @ M.T
flux_samples = central_flux * (1 + rel_var)  # (n_samples, 2400)

# Extract H group proton and neutron fluxes for plotting
mc_mean_p = np.mean(flux_samples[:, :n_energy], axis=0)
mc_std_p = np.std(flux_samples[:, :n_energy], axis=0)
mc_mean_n = np.mean(flux_samples[:, n_energy:rpg], axis=0)
mc_std_n = np.std(flux_samples[:, n_energy:rpg], axis=0)

fig, ax = plt.subplots(figsize=(6, 3))

for fl in flux_samples[::2]:
    ax.loglog(xkin, xkin**E * fl[:n_energy], color="C0", alpha=0.08, lw=0.5)
    ax.loglog(xkin, xkin**E * fl[n_energy:rpg], color="C1", alpha=0.08, lw=0.5)

ax.loglog(xkin, xkin**E * cf_p_H, "C0", lw=2, label="Protons (central)")
ax.loglog(xkin, xkin**E * cf_n_H, "C1", lw=2, ls="--", label="Neutrons (central)")
ax.fill_between(
    xkin,
    xkin**E * (mc_mean_p - mc_std_p),
    xkin**E * (mc_mean_p + mc_std_p),
    color="C0",
    alpha=0.8,
)
ax.fill_between(
    xkin,
    xkin**E * (mc_mean_n - mc_std_n),
    xkin**E * (mc_mean_n + mc_std_n),
    color="C1",
    alpha=0.8,
)

ax.set_xlabel("Kinetic energy per nucleon (GeV)")
ax.set_ylabel(r"$E^{2.7} \times \Phi$")
ax.set_xlim(1, 1e11)
ax.set_ylim(1, 1e5)
ax.legend()
plt.tight_layout()

## Neutron-to-proton ratio

The ratio $n/p$ matters for atmospheric neutrino calculations and composition studies. Its uncertainty comes from the correlated 12-component part only (the $M\phi$ term).

In [ ]:
flux_smooth = central_flux * (
    1 + phi @ M.T
)  # smooth 12-component part only (no diagonal eps)

# Total proton and neutron fluxes (summed over all 4 groups)
total_p_samples = sum(
    flux_smooth[:, ig * rpg : ig * rpg + n_energy] for ig in range(n_groups)
)
total_n_samples = sum(
    flux_smooth[:, ig * rpg + n_energy : (ig + 1) * rpg] for ig in range(n_groups)
)

ratio_samples = total_n_samples / total_p_samples
ratio_mean = np.mean(ratio_samples, axis=0)
ratio_std = np.std(ratio_samples, axis=0)

fig, ax = plt.subplots(figsize=(9, 4))

for r in ratio_samples[::10]:
    ax.semilogx(xkin, r, color="0.6", alpha=0.05, lw=0.5)

ax.semilogx(xkin, ratio_mean, "k", lw=2, label="Mean $n/p$ ratio")
ax.fill_between(
    xkin,
    ratio_mean - ratio_std,
    ratio_mean + ratio_std,
    color="k",
    alpha=0.25,
    label=r"$\pm 1\sigma$",
)

ax.set_xlabel("Kinetic energy per nucleon (GeV)")
ax.set_ylabel("Neutron / Proton flux ratio")
ax.set_title("Neutron-to-proton ratio (total, all groups)")
ax.set_ylim(-0.05, 0.65)
ax.legend()
plt.tight_layout()

## Varying individual components

Set selected $\phi_k$ to non-zero values and visualize the resulting flux variation (H group). The shaded grey region in the lower panels is the full $\pm 1\sigma$ envelope from the model.

In [ ]:
from matplotlib.ticker import LogLocator

# ±1σ relative uncertainty from the Hybrid PCA model (H group)
sigma_rel_p = np.sqrt(np.sum(M[:n_energy] ** 2, axis=1) + D[:n_energy])
sigma_rel_n = np.sqrt(np.sum(M[n_energy:rpg] ** 2, axis=1) + D[n_energy:rpg])

E_break1 = 1e4
E_break2 = 1e9

fig = plt.figure(figsize=(12, 7))
gs = fig.add_gridspec(
    2,
    3,
    height_ratios=[3, 1.8],
    hspace=0.06,
    width_ratios=[
        np.log10(E_break1),
        np.log10(E_break2 / E_break1),
        np.log10(1e11 / E_break2),
    ],
    wspace=0.12,
)

ax_main = fig.add_subplot(gs[0, :])
ax_r1 = fig.add_subplot(gs[1, 0])
ax_r2 = fig.add_subplot(gs[1, 1])
ax_r3 = fig.add_subplot(gs[1, 2])

# Central flux for H group
ax_main.loglog(xkin, xkin**E * cf_p_H, "k", lw=2, label="Central (proton)")
ax_main.loglog(xkin, xkin**E * cf_n_H, "k--", lw=2, label="Central (neutron)")

sigma_rel_tot = np.maximum(sigma_rel_p, sigma_rel_n)
for ax_r in (ax_r1, ax_r2, ax_r3):
    ax_r.axhline(1, color="k", lw=0.8, ls=":")
    ax_r.fill_between(
        xkin, 1 - sigma_rel_tot, 1 + sigma_rel_tot, color="0.75", alpha=0.5, zorder=0
    )

# Vary individual components using M directly: f = f0 * (1 + delta * M_k)
for k, (delta, ls) in enumerate([(+3, "-"), (-1, "--"), (+2, "-"), (-1, "--")]):
    varied_p = cf_p_H * (1 + delta * M[:n_energy, k])
    varied_n = cf_n_H * (1 + delta * M[n_energy:rpg, k])
    sign = "+" if delta > 0 else ""
    col = ax_main.loglog(
        xkin,
        xkin**E * varied_p,
        ls=ls,
        label=rf"$\phi_{{{k + 1}}} = {sign}{delta}$",
    )[0].get_color()
    ax_main.loglog(xkin, xkin**E * varied_n, color=col, ls=ls)
    for ax_r in (ax_r1, ax_r2, ax_r3):
        ax_r.semilogx(xkin, varied_p / cf_p_H, color=col, ls="-", lw=1.2)
        ax_r.semilogx(xkin, varied_n / cf_n_H, color=col, ls="--", lw=1.2)

ax_main.set_ylabel(r"$E^{2.7} \times \Phi$")
ax_main.set_title("Component variations (solid = protons, dashed = neutrons)")
ax_main.legend(fontsize=8, ncols=3)
ax_main.set_xlim(1, 1e11)
ax_main.set_ylim(50.0, 3e4)
ax_main.tick_params(labelbottom=False)

ax_r1.set_xlim(1, E_break1)
ax_r1.set_xscale("log")
ax_r1.set_xlabel("Energy (GeV)")
ax_r1.set_ylabel("Ratio to central")
ax_r1.set_ylim(0.96, 1.04)
ax_r1.grid(True, which="both", ls=":", lw=0.5, alpha=0.5)

ax_r2.set_xlim(E_break1, E_break2)
ax_r2.set_xscale("log")
ax_r2.set_xlabel("Energy (GeV)")
ax_r2.tick_params(axis="y", direction="in", pad=-30, labelleft=True)
ax_r2.set_ylim(0.5, 1.5)
ax_r2.grid(True, which="both", ls=":", lw=0.5, alpha=0.5)

ax_r3.set_xlim(E_break2, 1e11)
ax_r3.set_xscale("log")
ax_r3.set_xlabel("Energy (GeV)")
ax_r3.yaxis.tick_right()
ax_r3.yaxis.set_label_position("right")
ax_r3.set_ylabel("Ratio to central")
ax_r3.grid(True, which="both", ls=":", lw=0.5, alpha=0.5)

for ax_r in (ax_r1, ax_r2, ax_r3):
    ax_r.xaxis.set_major_locator(LogLocator(base=10, numticks=5))

plt.tight_layout()

## Gradients at arbitrary energies

Reduced Jacobians evaluate exactly at any energy grid via $M(E) = J_{\text{rel}}(E) L_{\text{param}}$ — useful for fitting on grids that differ from the construction grid.

In [ ]:
E_fit = np.logspace(1, 8, 30)  # 30 arbitrary fit points
M_fit = pca.reduced_jacobian(E_fit)  # exact at any energy, no interpolation

print(f"Reduced Jacobian at {len(E_fit)} fit energies: {M_fit.shape}")
print(
    f"Row-norm range: [{np.min(np.linalg.norm(M_fit, axis=1)):.3f},"
    f" {np.max(np.linalg.norm(M_fit, axis=1)):.3f}]  (O(1), no steep-spectrum issue)"
)

# Condition number check
print(f"Condition number of M_fit: {np.linalg.cond(M_fit):.1f}  (well-conditioned)")

## Cutoff-aware uncertainty propagation

The decomposition is a property of the **spline parameters**, so the cutoff enters only through the Jacobian: $M(E, R_{\text{cut}}) = J_{\text{rel}}(E, R_{\text{cut}}) L_{\text{param}}$.

A sharp Heaviside cutoff would blow up the relative Jacobian where $f_{\text{cut}} \to 0$. Use `cutoff_width > 0` for a smooth sigmoid modeling the geomagnetic penumbra.

In the transition region (~1–100 GeV/nucleon), the composition shifts because protons ($A/Z = 1$) are suppressed more than heavier nuclei ($A/Z \approx 2$). Above ~100 GeV/nucleon the curves match the no-cutoff case.

In [ ]:
R_CUT = 20.0  # GV (typical equatorial cutoff)
E_plot = np.logspace(0, 7, 300)
en_exp = 3.0

# Smooth cutoff model (width = 2 GV penumbral transition)
gsf_smooth = GSFEnergyPerNucleon(
    version="2025",
    default_time_interval=(200812, 202001),
    use_approximate_solar_cycle_average=True,
    default_rigidity_cutoff=R_CUT,
    cutoff_width=1.0,
)

f_nocut = gsf.total_flux(E_plot)
f_smooth = gsf_smooth.total_flux(E_plot)
err_nocut = gsf.total_error(E_plot)
err_smooth = gsf_smooth.total_error(E_plot)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

ax1.plot(E_plot, f_nocut * E_plot**en_exp, "k-", lw=2, label="No cutoff")
ax1.fill_between(
    E_plot,
    (f_nocut - err_nocut) * E_plot**en_exp,
    (f_nocut + err_nocut) * E_plot**en_exp,
    facecolor="k",
    alpha=0.1,
)
ax1.plot(
    E_plot,
    f_smooth * E_plot**en_exp,
    "C0-",
    lw=2,
    label=f"$R_{{cut}}$ = {R_CUT} GV (width = 2 GV)",
)
ax1.fill_between(
    E_plot,
    (f_smooth - err_smooth) * E_plot**en_exp,
    (f_smooth + err_smooth) * E_plot**en_exp,
    facecolor="C0",
    alpha=0.15,
)
ax1.set_xscale("log")
ax1.set_yscale("log")
ax1.set_xlabel(r"$E_N$ [GeV/nucleon]")
ax1.set_ylabel(f"Nucleon Flux x $(E_N/\\mathrm{{GeV}})^{{{en_exp}}}$")
ax1.set_xlim(1, 1e7)
ax1.set_ylim(5e2, 1e7)
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_title("Total nucleon flux with smooth cutoff")

# Relative uncertainty with per-group breakdown (dashed)
with np.errstate(divide="ignore", invalid="ignore"):
    rel_nocut = err_nocut / np.maximum(f_nocut, 1e-90)
    rel_smooth = np.where(f_smooth > 1e-90, err_smooth / f_smooth, np.nan)

ax2.plot(E_plot, rel_nocut, "k-", lw=2.5, label="No cutoff (total)")
valid = f_smooth > 1e-90
ax2.plot(E_plot[valid], rel_smooth[valid], "C0-", lw=2.5, label="Smooth cutoff (total)")

# Per-group uncertainties (dashed, distinct colors)
for name, color, label in [
    ("p", "r", "p"),
    ("He", "orange", "He"),
    ("O*", "green", "O*"),
    ("Fe*", "purple", "Fe*"),
]:
    fg = gsf.flux(E_plot, name)
    eg = gsf.error(E_plot, name)
    with np.errstate(divide="ignore", invalid="ignore"):
        rg = eg / np.maximum(fg, 1e-90)
    ax2.plot(E_plot, rg, "--", color=color, lw=1, alpha=0.5, label=f"{label} (group)")

ax2.set_xscale("log")
ax2.set_yscale("log")
ax2.set_xlabel(r"$E_N$ [GeV/nucleon]")
ax2.set_ylabel("Relative uncertainty")
ax2.set_xlim(1, 1e7)
ax2.set_ylim(3e-3, 1)
ax2.legend(fontsize=8, ncol=2)
ax2.grid(True, alpha=0.3)
ax2.set_title("Relative uncertainty (per-group dashed, totals solid)")

plt.tight_layout()
plt.show()

### Sharp vs. smooth cutoff in $M(E)$

Reduced Jacobian components for sharp (red) and smooth (blue, 2 GV width) cutoffs. The sharp cutoff produces an unphysical spike at the transition; the smooth one stays well-behaved. Always prefer `cutoff_width > 0` for practical use.

In [ ]:
# Build cutoff-aware reduced Jacobian
gsf_sharp = GSFEnergyPerNucleon(
    version="2025",
    default_time_interval=(200812, 202001),
    use_approximate_solar_cycle_average=True,
    default_rigidity_cutoff=R_CUT,
)

pca_nocut = HybridPCA(gsf, n_components=12, energy_grid=xkin)
pca_sharp = HybridPCA(gsf_sharp, n_components=12, energy_grid=xkin)
pca_smooth = HybridPCA(gsf_smooth, n_components=12, energy_grid=xkin)

M_nocut = pca_nocut.reduced_jacobian(E_plot)
M_sharp = pca_sharp.reduced_jacobian(E_plot)
M_smooth = pca_smooth.reduced_jacobian(E_plot)
n_e = len(E_plot)

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True)

for k, ax in enumerate(axes.flatten()):
    # H group proton rows (first n_e rows of the stacked layout)
    ax.plot(E_plot, M_nocut[:n_e, k], "k-", lw=1.5, alpha=0.4, label="No cutoff")
    ax.plot(E_plot, M_sharp[:n_e, k], "C3-", lw=1.2, alpha=0.7, label="Sharp cutoff")
    ax.plot(E_plot, M_smooth[:n_e, k], "C0-", lw=1.5, label="Smooth (w=2 GV)")
    ax.set_xscale("log")
    ax.set_xlim(1, 1e7)
    ax.set_title(f"Component {k + 1} (H group proton nucleons)", fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color="gray", ls="-", lw=0.5, alpha=0.3)
    if k == 0:
        ax.legend(fontsize=7)
    if k >= 3:
        ax.set_xlabel(r"$E_N$ [GeV/nucleon]")
    if k % 3 == 0:
        ax.set_ylabel(r"$M_{ij}$ (relative)")

fig.suptitle(
    f"Reduced Jacobian: sharp vs smooth cutoff ($R_{{{{cut}}}}$ = {R_CUT} GV)", y=1.01
)
plt.tight_layout()
plt.show()

## Appendix: Hybrid PCA with `GSFEnergy`

For non-nucleon models there's no p/n split: the system has $4 \times N$ rows, no $D_{pn}$, and the diagonal $D$ alone preserves variances.

In [ ]:
from globalsplinefit import GSFEnergy
from globalsplinefit.pca import HybridPCA as HybridPCA_  # avoid shadowing

gsf_energy = GSFEnergy(version="2025")
pca_energy = HybridPCA_(gsf_energy, n_components=12)

E_en = np.logspace(2, 7, 200)  # 100 GeV to 10 PeV (total energy per nucleus)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: flux with uncertainty bands
ax = axes[0]
for g, col in [("p", "C0"), ("He", "C1"), ("O*", "C2"), ("Fe*", "C3")]:
    flux = gsf_energy.flux(E_en, g)
    pca_err = pca_energy.error(E_en, g)
    ax.loglog(E_en, E_en**2.7 * flux, color=col, lw=2, label=g)
    ax.fill_between(
        E_en,
        E_en**2.7 * (flux - pca_err),
        E_en**2.7 * (flux + pca_err),
        color=col,
        alpha=0.15,
    )
ax.set_xlabel("Total energy per nucleus (GeV)")
ax.set_ylabel(r"$E^{2.7} \times \Phi$")
ax.set_title(r"GSFEnergy — flux with PCA $\pm 1\sigma$ bands")
ax.set_xlim(E_en[0], E_en[-1])
ax.legend()

# Right: PCA/exact error ratio
ax = axes[1]
for g, col in [("p", "C0"), ("He", "C1"), ("O*", "C2"), ("Fe*", "C3")]:
    pca_err = pca_energy.error(E_en, g)
    model_err = gsf_energy.error(E_en, g)
    ax.semilogx(E_en, pca_err / model_err, color=col, lw=1.5, label=g)
ax.axhline(1.0, color="k", ls="--", lw=1)
ax.set_xlabel("Total energy per nucleus (GeV)")
ax.set_ylabel("PCA / Exact")
ax.set_title("Error ratio (identically 1)")
ax.set_ylim(0.99, 1.01)
ax.legend()

plt.tight_layout()
print(f"Variance explained: {pca_energy.variance_explained:.1%}")

In [ ]:
# Monte Carlo samples with GSFEnergy — simpler than nucleon models (no p/n split)
E_mc = np.logspace(2, 6, 200)
samples_en = pca_energy.sample(500, E_mc, rng=np.random.default_rng(0))
n_mc = len(E_mc)
print(f"Sample shape: {samples_en.shape}  (500 samples x 4 groups x {n_mc} energies)")

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for ax, ig, g, col in [
    (axes[0, 0], 0, "p", "C0"),
    (axes[0, 1], 1, "He", "C1"),
    (axes[1, 0], 2, "O*", "C2"),
    (axes[1, 1], 3, "Fe*", "C3"),
]:
    samp = samples_en[:, ig * n_mc : (ig + 1) * n_mc]
    central = gsf_energy.flux(E_mc, g)
    # Show a handful of individual smooth realisations
    for s in samp[:5]:
        ax.loglog(E_mc, E_mc**2.7 * s, color=col, alpha=0.4, lw=0.8)
    ax.loglog(E_mc, E_mc**2.7 * central, "k", lw=2, label="Central")
    ax.fill_between(
        E_mc,
        E_mc**2.7 * np.percentile(samp, 16, axis=0),
        E_mc**2.7 * np.percentile(samp, 84, axis=0),
        color=col,
        alpha=0.2,
        label=r"$\pm 1\sigma$ (MC)",
    )
    ax.set_xlabel("Total energy per nucleus (GeV)")
    ax.set_ylabel(r"$E^{2.7} \times \Phi$")
    ax.set_title(f"{g} group — MC samples")
    ax.legend(fontsize=8)
    ax.set_xlim(E_mc[0], E_mc[-1])

plt.tight_layout()